# The HS risk index

The HS index is a risk index in its own right, distinct from the Tuckman-Chang
score in `02-features-and-benchmarks`. It is built on its own ratio set — twelve
ratios (`features.HS_RATIOS`) — organised into five financial dimensions:
solvency, liquidity, profitability, efficiency and revenue. Each charity is
placed against the previous year's sector on every ratio, the ratios are averaged
within their dimension, and the five dimensions are averaged into one score.

Tuckman and Chang appear here only as a **benchmark**. Their score is a
different index — four ratios, one quintile flag each, counted to 0-4 — and
`02-features-and-benchmarks` found it sorts risk but wastes most of the signal.
The comparison is a natural one, because the two indices share four ratios and
both use fixed reference points with no fitted weights. But the HS index is not
a revision of that rule; the questions this chapter asks are whether the HS
index separates charities that deregister from those that do not, and how it
compares with the Tuckman-Chang score and the fitted models in
`03-model-results`. The index's formal statement sits with the other models'
in that notebook's specification section.

In [ ]:
import logging
import warnings
warnings.filterwarnings("ignore")
# The index logs ratios it cannot score; this chapter reports them itself.
logging.getLogger("charity_risk").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from charity_risk import config, plots
from charity_risk.benchmarks import TuckmanChangScore
from charity_risk.dataset import exit_split, load_dataset
from charity_risk.evaluate import (
    bootstrap_metric_difference, compare_models, fit_all, lift_table,
)
from charity_risk.features import HS_RATIOS, TUCKMAN_CHANG
from charity_risk.models import SPECIFICATIONS
from charity_risk.hs_index import (
    HS_DIMENSIONS, HS_RISK_DIRECTION, HSRiskIndex,
)

plots.use_project_style()
pd.set_option("display.width", 200)

# The transparent end of the model ladder, compared in Section 6.
keys = ["size_only", "tuckman_chang_1991", "hs_index_flags", "hs_index",
        "greenlee_trussel_2000", "trussel_2002"]
columns = sorted({c for key in keys for c in SPECIFICATIONS[key].columns}
                 | set(HS_RATIOS) | set(TUCKMAN_CHANG)) + ["exit_next_year", "is_section_d"]
frame = load_dataset(years=[config.TRAIN_YEAR, config.TEST_YEAR], columns=columns)
train, test = exit_split(frame)
y_test = test["exit_next_year"].to_numpy()
print(f"train {config.TRAIN_YEAR}: {len(train):,} rows, exit rate {train['exit_next_year'].mean():.2%}")
print(f"test  {config.TEST_YEAR}: {len(test):,} rows, exit rate {test['exit_next_year'].mean():.2%}")

## 1. Twelve ratios in five dimensions

| Dimension | Ratio | Construction | Risky tail |
|---|---|---|---|
| Solvency | `equity_balance` | net assets / revenue | low |
| | `net_assets_to_assets` | net assets / total assets | low |
| | `equity_ratio` | net assets / total liabilities | low |
| Liquidity | `liquidity_ratio` | working capital / total assets | low |
| | `working_capital_months` | 12 × working capital / total expenditures | low |
| Profitability | `return_on_assets` | surplus / total assets | low |
| | `net_revenue` | revenue − expenditures (dollars) | low |
| | `operating_margin` | surplus / revenue | low |
| | `operating_markup` | surplus / expenditures | low |
| Efficiency | `admin_cost_ratio` | management and administration / expenditures before gifts | **low** |
| Revenue | `revenue_growth_volatility` | expanding s.d. of log revenue growth | high |
| | `revenue_concentration` | Herfindahl over the eight revenue buckets | high |

Three of the twelve behave differently from the rest, and the index has to
account for each.

*The liquidity ratios exist only on Schedule 6.* Working capital needs current
assets and liabilities, which the short Section D form never asks for. So a third
of the register has no liquidity dimension at all.

*Revenue volatility needs history.* It is a standard deviation of at least two
growth rates, so it needs three consecutive returns — which no charity in the
2020 training year has, in a window that starts in 2019.

*Net revenue is in dollars.* Its bottom tail is large deficits, which mostly
belong to large charities. It is kept because it is in the specification, but it
is as much a measure of scale as of profitability.

In [ ]:
by_tier = train.groupby("is_section_d")[list(HS_RATIOS)].apply(lambda d: d.isna().mean()).T
by_tier.columns = ["missing: Schedule 6", "missing: Section D"]
dimension_of = {r: d for d, ratios in HS_DIMENSIONS.items() for r in ratios}
summary = train[list(HS_RATIOS)].describe(percentiles=[.2, .5, .8]).T[["20%", "50%", "80%"]]
summary.insert(0, "dimension", pd.Series(dimension_of))
summary["risky_tail"] = [("low" if HS_RISK_DIRECTION[r] < 0 else "high") for r in summary.index]
summary.join(by_tier).round(3)

## 2. Why weight dimensions, not ratios

If the ratios inside a dimension measure the same thing, a flat count gives that
thing several votes. The HS index therefore weights dimensions, not ratios. The
rank correlations below make the case: read the blocks
on the diagonal (within a dimension) against the rest of the matrix.

In [ ]:
ordered = [r for ratios in HS_DIMENSIONS.values() for r in ratios]
rho = train[ordered].corr(method="spearman")

within, between = [], []
for i, a in enumerate(ordered):
    for b in ordered[i + 1:]:
        (within if dimension_of[a] == dimension_of[b] else between).append(abs(rho.loc[a, b]))
print(f"mean |Spearman rho| within a dimension : {np.nanmean(within):.2f}")
print(f"mean |Spearman rho| between dimensions : {np.nanmean(between):.2f}")
rho.round(2)

The profitability block is the clearest: operating margin, markup and return on
assets share a numerator, so they move together almost by construction. Under a
flat twelve-ratio count, profitability alone would supply a third of the flags.
Averaging within the dimension first gives it one vote in five.

## 3. Constructing the index

`HSRiskIndex` scores every ratio against the 2020 distribution and
applies those reference points, unchanged, to 2021. Two per-ratio scores are
available:

* **flags** — 1 inside the ratio's risky fifth, 0 outside, so the index is the
  dimension-weighted share of flags;
* **percentile** — the share of the 2020 sector that looked safer on that ratio,
  so 0.95 means only one charity in twenty was worse.

Within each dimension the scores are averaged, and the index is the average of
the dimensions that are observed. A dimension with no ratio observed is left out
rather than scored as safe, so a Section D filer is scored on four dimensions,
not penalised for lacking the fifth. Missing is never a flag.

`predict_proba` passes the index through a one-variable logit fitted on 2020.
That is monotone, so it changes no ranking; it only puts the index on the
probability scale the other models are measured on.

In [ ]:
percentile = HSRiskIndex(method="percentile").fit(train, train["exit_next_year"])
flags = HSRiskIndex(method="flags").fit(train, train["exit_next_year"])
print("ratios not scored (missing throughout 2020):", percentile.unscored_ratios_)

pd.DataFrame({
    "dimension": pd.Series(dimension_of),
    "risky_tail": pd.Series({r: ("bottom quintile" if d < 0 else "top quintile")
                             for r, d in HS_RISK_DIRECTION.items()}),
    f"cutpoint_{config.TRAIN_YEAR}": pd.Series(flags.cutpoints_),
}).loc[ordered].round(4)

In [ ]:
test_scored = test.assign(
    hs_index=percentile.index(test).to_numpy(),
    hs_index_flags=flags.index(test).to_numpy(),
    hs_dimensions_at_risk=percentile.n_dimensions_at_risk(test).to_numpy(),
    hs_dimensions_scored=percentile.n_dimensions_scored(test).to_numpy(),
    tc_score=TuckmanChangScore().fit(train[list(TUCKMAN_CHANG)]).score(test[list(TUCKMAN_CHANG)]).to_numpy(),
)
fig, ax = plots.plt.subplots(figsize=(7.0, 4.0))
ax.hist(test_scored["hs_index"].dropna(), bins=60, color=plots.series_color(0))
ax.set_xlabel("HS risk index (percentile method)")
ax.set_ylabel("charities")
ax.set_title(f"The index across the {config.TEST_YEAR} register");

## 4. Does it sort risk?

First as a count: a dimension is at risk when its average percentile sits in
the riskiest fifth, and the exit rate is shown at each number of dimensions at
risk. The benchmark Tuckman-Chang 0-4 score is set beside it on the same
charities.

In [ ]:
fig, axes = plots.plt.subplots(1, 2, figsize=(13.0, 4.4))
plots.plot_score_response(test_scored, "tc_score", "exit_next_year", ax=axes[0],
                          title="Benchmark: Tuckman-Chang score (0-4)")
plots.plot_score_response(test_scored, "hs_dimensions_at_risk", "exit_next_year", ax=axes[1],
                          title="HS index: dimensions at risk (0-5)");

In [ ]:
response = test_scored.groupby("hs_dimensions_at_risk").agg(
    n_charities=("exit_next_year", "size"),
    exit_rate=("exit_next_year", "mean"),
    mean_dimensions_scored=("hs_dimensions_scored", "mean"),
)
response["share_of_register"] = response["n_charities"] / len(test_scored)
response["lift"] = response["exit_rate"] / test_scored["exit_next_year"].mean()
response.round(3)

What to look for: whether the exit rate rises at every step, rather than only at
the top, and whether the bottom two counts swallow most of the register — the
two ways the Tuckman-Chang score fell short in `02-features-and-benchmarks`.

The count is still a discretisation. The continuous index is the version meant
for ranking, so the decile table below is the fairer test of what the dimensions
carry.

In [ ]:
lift = lift_table(y_test, test_scored["hs_index"].fillna(percentile.base_rate_).to_numpy())
fig, ax = plots.plt.subplots(figsize=(7.0, 4.2))
plots.plot_decile_lift(lift, ax=ax, title="Lift by decile of the HS index")
lift.round(3)

## 5. Which dimensions carry the signal

Each dimension score, on its own, as a ranking of 2021 exits. A dimension with
an AUC below 0.5 is pointing the wrong way on this outcome.

In [ ]:
dims = percentile.dimension_scores(test)
rows = []
for dimension in HS_DIMENSIONS:
    observed = dims[dimension].notna().to_numpy()
    rows.append({
        "dimension": dimension,
        "coverage": observed.mean(),
        "roc_auc": roc_auc_score(y_test[observed], dims.loc[observed, dimension]),
        "average_precision": average_precision_score(y_test[observed],
                                                     dims.loc[observed, dimension]),
        "base_rate_where_observed": y_test[observed].mean(),
    })
pd.DataFrame(rows).set_index("dimension").round(3)

Two readings are available in advance from the earlier chapters. Efficiency is a
single ratio, the administrative cost ratio, whose risky tail the HS index puts at
the *bottom* but which `02-features-and-benchmarks` finds predicts exit at
the *top*; expect it below 0.5. And the revenue dimension is scored on
concentration alone in this split, since volatility had no 2020 values to set
its reference points.

## 6. Against the benchmarks

The same out-of-time comparison as `03-model-results`, restricted to the
transparent end of the ladder: both forms of the HS index against three
benchmarks — the size-only logit, the Tuckman-Chang score, and the two classical
logits.

In [ ]:
fitted = fit_all(train, "exit_next_year", specs={k: SPECIFICATIONS[k] for k in keys})
results = compare_models(fitted, test, "exit_next_year")
results[["label", "roc_auc", "average_precision", "lift_top1pct", "lift_top5pct",
         "recall_top5pct", "calibration_slope"]].round(3)

In [ ]:
risks = {key: fitted[key].risk(test) for key in keys}
rows = []
for versus in ["tuckman_chang_1991", "trussel_2002"]:
    for metric_name, metric in [("roc_auc", roc_auc_score),
                                ("average_precision", average_precision_score)]:
        stats = bootstrap_metric_difference(y_test, risks["hs_index"], risks[versus],
                                            metric=metric)
        rows.append({"hs_index versus": versus, "metric": metric_name, **stats})
pd.DataFrame(rows).round(4)

The paired bootstrap resamples the same charities for both scores, so the
interval is on the *difference*. An interval above zero against the
Tuckman-Chang score says the HS index ranks exits better than that benchmark;
the comparison with Trussel (2002) asks whether an
unweighted index can stand in for a fitted logit with a size control and sector
effects.

## 7. The administrative cost ratio, flipped

The HS index treats a *low* administrative cost ratio as risky — spending with
no slack left to cut. On this outcome that is the wrong tail. Refitting with the direction
reversed shows how much the specification choice costs.

In [ ]:
flipped = HSRiskIndex(directions={"admin_cost_ratio": +1}).fit(
    train, train["exit_next_year"])
risk_flipped = flipped.predict_proba(test)[:, 1]
pd.DataFrame({
    "roc_auc": [roc_auc_score(y_test, risks["hs_index"]), roc_auc_score(y_test, risk_flipped)],
    "average_precision": [average_precision_score(y_test, risks["hs_index"]),
                          average_precision_score(y_test, risk_flipped)],
}, index=["as specified (low ACR risky)", "flipped (high ACR risky)"]).round(3)

## 8. What the index is, and is not

**It is transparent.** Every score decomposes into five dimension scores and
twelve percentiles, each of which says where the charity sits against last
year's sector. A reviewer can read *why* a charity is flagged, which no fitted
model in `03-model-results` offers as directly.

**It is not fitted.** No weight is estimated, so every dimension counts the same
whether or not it predicts exit, and a mis-signed ratio (Section 7) degrades the
index rather than being corrected by it. The price of transparency is measured
in Section 6.

**It inherits the sample's limits.** The liquidity dimension exists for Schedule
6 filers only; revenue volatility cannot be scored from a 2020 training year;
and the outcome is deregistration, which bundles failure together with solvent
wind-ups and amalgamations. Re-fitting on a later training year, once three
consecutive returns are available, brings volatility into the revenue dimension.